# 2 — Detector test against the base pattern

This notebook gives the output for the paper sections **Evaluation** and **Results → "Detector test against the base pattern"**: the tables `tab:base` and `tab:recolor` and the figure `fig_composites`.

Each site-specific pattern from notebook 1 is compared with the base pattern it was made from, on the same scene, at the same position, with the same target shape: 33 scenes × 3 shapes × 20 positions × 2 conditions × 3 base patterns × 2 detectors = 23,760 detector runs. A lower $S_\alpha$ means that the detector found the target less well, so a negative difference favors the site-specific pattern.

**Start notebook 1 first** (it makes the patterns), and start this notebook from the project root. The detector run takes several hours on the Apple GPU.

**Names.** `woodland` in the code and the CSV files is the base pattern that the paper calls **M81**.

**Outputs:** `results/02_adversarial_test/IMG_4171_cal/results.csv`, `figures/`, `tables/`.

## Research question: hypothesis half (b)

**Hypothesis** (proposal wording). "While k-means quantization restricts color variance for physical printability, the structural blending from Neural Style Transfer will sufficiently mitigate this penalty, maintaining a significant advantage over generic baselines."

**Half (b)** is the second part: "maintaining a significant advantage over generic baselines". In this study, the baseline of each site-specific pattern is the base pattern it was made from, brought to the size of the site-specific pattern (cropped to the shape 3:4, then reduced with `cv2.INTER_AREA` to 400 × 300), so that the two differ in design and not in image size.

*Measure:* $\Delta S_\alpha$ = site-specific − base for each base pattern and detector, paired over 33 backgrounds.
*Rule:* half (b) is supported in a cell when $\Delta S_\alpha < 0$ and the Holm-corrected two-sided Wilcoxon $p < 0.05$. A significant $\Delta S_\alpha > 0$ is a result in the **wrong** direction, not a missing effect. The pre-registered margin $\varepsilon = 0.03$ tells whether an effect is large enough to matter.

The last cell of this notebook applies these rules.

In [ ]:
import os, sys, csv, json, shutil, hashlib
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')   # let the detectors use the Apple GPU
from collections import defaultdict
from pathlib import Path
import numpy as np, cv2
import matplotlib.pyplot as plt
from scipy import stats
ROOT = Path.cwd()
assert (ROOT/'images').is_dir(), 'start this notebook from the project root'

# ==================== SETTINGS ====================
FRAME     = 'IMG_4171'                          # generation frame (must match notebook 1)
TAG       = '_cal'
STYLES    = ['multicam', 'woodland', 'realtree']
NAME      = {'multicam': 'MultiCam', 'woodland': 'M81', 'realtree': 'Realtree'}   # 'woodland' is the M81 base
DETECTORS = ('SINet-V2', 'DGNet')
DEVICE    = 'mps'                               # 'mps' (Apple GPU) | 'cpu'
N_TRIALS  = 20                                  # placements per background and shape
BG_LIMIT  = None                                # None = all 33 held-out backgrounds
FRESH_RUN = True                                # True: erase the earlier results first | False: continue an interrupted run
EPS       = 0.03                                # pre-registered practical-significance margin for S_alpha

PNG_DIR   = ROOT/'images'/'site_photos'/'calibrated'/'png'
PATTERNS  = ROOT/'results'/'01_generation'/'patterns'
OUT       = ROOT/'results'/'02_adversarial_test'
EVAL      = OUT/(FRAME+TAG)
FIG       = OUT/'figures'
# backgrounds photographed at the generation location (IMG_4171-4174, 4174 is a chart frame);
# every other held-out background is from the second location, 11.3 km away
LOCATION_A = {'IMG_4172', 'IMG_4173'}

for style in STYLES:
    assert (PATTERNS/style/(FRAME+TAG)/'final_camo.png').exists(), f'{style}: pattern missing, start notebook 1 first'
if FRESH_RUN and OUT.exists():
    shutil.rmtree(OUT)
FIG.mkdir(parents=True, exist_ok=True)
print(f'styles={STYLES}  device={DEVICE}  n_trials={N_TRIALS}  out={OUT.relative_to(ROOT)}')

## Code

The cells of this section hold all the code that the steps below use. Each code cell is one module, and the text above it names the section of the paper that it implements. Run this section before the steps.

In [ ]:
# A cell with %%module on its first line holds one module of the code. The cell runs as its
# own Python module, so its names do not mix with the names of the notebook (both can have OUT,
# for example). Other cells get a module with a usual import, for example `import adversarial_eval as rv`.
import sys, types, linecache
from IPython.core.magic import register_cell_magic

class CellSource:
    """Gives the cell text to linecache again, so that tracebacks and TensorFlow can read it."""
    def __init__(self, text):
        self.text = text

    def get_source(self, name):
        return self.text

@register_cell_magic
def module(line, cell):
    name = line.strip()
    m = types.ModuleType(name)
    m.__file__ = f'%%module {name}'                    # the file name in tracebacks
    m.__loader__ = CellSource(cell)
    linecache.cache[m.__file__] = (len(cell), None, cell.splitlines(True), m.__file__)
    sys.modules[name] = m
    exec(compile(cell, m.__file__, 'exec', dont_inherit=True), m.__dict__)

### Target shapes — Evaluation, "Digital Evaluation"

The three silhouettes that are pasted into the test scenes. The human silhouette is 0.35 of the frame height; the animal and vehicle bodies are 0.30 and 0.35 of the frame width.

In [ ]:
%%module silhouettes
"""The three target shapes that are pasted into the test scenes.

The human silhouette is 0.35 of the frame height; the animal and vehicle bodies are 0.30
and 0.35 of the frame width. Each function returns a uint8 mask (255 = target).
"""
import numpy as np
import cv2


class SilhouetteLibrary:
    CLASSES = ['quadruped', 'human', 'vehicle']

    @classmethod
    def quadruped(cls, h, w, scale=0.30):
        m = np.zeros((h, w), dtype=np.uint8)
        cy = int(h * 0.62)
        body_w = int(w * scale)
        body_h = int(body_w * 0.45)
        cx = w // 2
        cv2.ellipse(m, (cx, cy), (body_w // 2, body_h // 2), 0, 0, 360, 255, -1)
        head_r = int(body_h * 0.55)
        head_cx = cx + body_w // 2 - head_r // 2
        head_cy = cy - body_h // 2 - head_r // 3
        cv2.circle(m, (head_cx, head_cy), head_r, 255, -1)
        leg_w = max(2, int(body_w * 0.06))
        leg_h = int(body_h * 0.95)
        for lx in (cx - body_w // 3, cx - body_w // 6, cx + body_w // 6, cx + body_w // 3):
            cv2.ellipse(m, (lx, cy + body_h // 3), (leg_w, leg_h // 2), 0, 0, 360, 255, -1)
        tail_pts = np.array([
            [cx - body_w // 2, cy - body_h // 4],
            [cx - body_w // 2 - int(body_w * 0.15), cy - body_h // 2],
            [cx - body_w // 2 - int(body_w * 0.12), cy - body_h // 4],
        ], dtype=np.int32)
        cv2.fillPoly(m, [tail_pts], 255)
        return m

    @classmethod
    def human(cls, h, w, scale=0.35):
        m = np.zeros((h, w), dtype=np.uint8)
        fig_h = int(h * scale)
        cx = w // 2
        cy = h // 2 + fig_h // 6
        head_r = max(4, int(fig_h * 0.08))
        head_cy = cy - fig_h // 2 + head_r
        cv2.circle(m, (cx, head_cy), head_r, 255, -1)
        torso_w = int(fig_h * 0.22)
        torso_h = int(fig_h * 0.35)
        torso_y0 = head_cy + head_r
        cv2.rectangle(m, (cx - torso_w // 2, torso_y0), (cx + torso_w // 2, torso_y0 + torso_h), 255, -1)
        arm_w = max(3, int(fig_h * 0.05))
        arm_h = int(fig_h * 0.32)
        cv2.rectangle(m, (cx - torso_w // 2 - arm_w, torso_y0), (cx - torso_w // 2, torso_y0 + arm_h), 255, -1)
        cv2.rectangle(m, (cx + torso_w // 2, torso_y0), (cx + torso_w // 2 + arm_w, torso_y0 + arm_h), 255, -1)
        leg_w = max(3, int(fig_h * 0.06))
        leg_h = int(fig_h * 0.40)
        leg_y0 = torso_y0 + torso_h
        cv2.rectangle(m, (cx - torso_w // 4 - leg_w, leg_y0), (cx - torso_w // 4, leg_y0 + leg_h), 255, -1)
        cv2.rectangle(m, (cx + torso_w // 4, leg_y0), (cx + torso_w // 4 + leg_w, leg_y0 + leg_h), 255, -1)
        return m

    @classmethod
    def vehicle(cls, h, w, scale=0.35):
        m = np.zeros((h, w), dtype=np.uint8)
        body_w = int(w * scale)
        body_h = int(body_w * 0.32)
        cx = w // 2
        cy = int(h * 0.60)
        body_y0 = cy - body_h // 2
        cv2.rectangle(m, (cx - body_w // 2, body_y0), (cx + body_w // 2, body_y0 + body_h), 255, -1)
        cab_pts = np.array([
            [cx - body_w // 4, body_y0],
            [cx - body_w // 6, body_y0 - int(body_h * 0.7)],
            [cx + body_w // 5, body_y0 - int(body_h * 0.7)],
            [cx + body_w // 3, body_y0],
        ], dtype=np.int32)
        cv2.fillPoly(m, [cab_pts], 255)
        wheel_r = int(body_h * 0.55)
        wheel_y = body_y0 + body_h
        cv2.circle(m, (cx - body_w // 3, wheel_y), wheel_r, 255, -1)
        cv2.circle(m, (cx + body_w // 3, wheel_y), wheel_r, 255, -1)
        return m

    @classmethod
    def get(cls, name, h, w):
        return getattr(cls, name)(h, w)

### Scores — Evaluation, "Digital Evaluation"

The paper uses the structure measure $S_\alpha$ (Fan et al. 2017), which compares the whole prediction map with the target mask. A lower $S_\alpha$ means that the detector found the target less well. The other scores ($E_\phi$, $F^w_\beta$, MAE) are stored in the result files but are not used in the paper.

In [ ]:
%%module cod_metrics
"""Detector scores. S_alpha (structure measure, Fan et al. 2017) is the score the paper uses.

    S_alpha - structure measure              (Fan et al., ICCV 2017)
    E_phi   - enhanced-alignment measure     (Fan et al., IJCAI 2018)
    Fw_beta - weighted F-measure             (Margolin et al., CVPR 2014)
    MAE     - mean absolute error

The functions are line-for-line ports of the MATLAB files in models/SINet-V2/eval/
(S_object.m, S_region.m, Enhancedmeasure.m, original_WFb.m). MATLAB std and var use N-1,
so every variance here uses ddof=1.
"""
import numpy as np
import cv2
from scipy.ndimage import distance_transform_edt as _bwdist

_EPS = np.finfo(np.float64).eps


def _matlab_round(v: float) -> int:
    """MATLAB round(): half away from zero. Python's round() is half to even."""
    return int(np.floor(v + 0.5)) if v >= 0 else int(np.ceil(v - 0.5))


class CODMetrics:
    # ------------------------------------------------------------------ MAE
    @staticmethod
    def mae(pred: np.ndarray, gt: np.ndarray) -> float:
        return float(np.mean(np.abs(pred.astype(np.float64) - gt.astype(np.float64))))

    # -------------------------------------------------------------- S-measure
    @staticmethod
    def _object(pred_region: np.ndarray) -> float:
        """S_object.m: score = 2x / (x^2 + 1 + sigma_x + eps)."""
        if pred_region.size == 0:
            return 0.0
        x = float(pred_region.mean())
        sigma_x = float(pred_region.std(ddof=1)) if pred_region.size > 1 else 0.0
        return float(2.0 * x / (x * x + 1.0 + sigma_x + _EPS))

    @classmethod
    def s_object(cls, pred: np.ndarray, gt: np.ndarray) -> float:
        fg = np.where(gt, pred, 0.0)
        bg = np.where(gt, 0.0, 1.0 - pred)
        u = float(gt.mean())
        return u * cls._object(fg[gt]) + (1.0 - u) * cls._object(bg[~gt])

    @staticmethod
    def _ssim(pred: np.ndarray, gt: np.ndarray) -> float:
        """S_region.m's ssim helper, sample variance (N-1)."""
        n = pred.size
        if n == 0:
            return 0.0
        if n == 1:
            # S_region.m: for N==1 all three variances are exactly 0, so alpha==0 and
            # beta==0 and the function takes its Q=1.0 branch regardless of the values.
            return 1.0
        x, y = float(pred.mean()), float(gt.mean())
        sx2 = float(((pred - x) ** 2).sum() / (n - 1 + _EPS))
        sy2 = float(((gt - y) ** 2).sum() / (n - 1 + _EPS))
        sxy = float(((pred - x) * (gt - y)).sum() / (n - 1 + _EPS))
        a = 4.0 * x * y * sxy
        b = (x * x + y * y) * (sx2 + sy2)
        if a != 0.0:
            return float(a / (b + _EPS))
        return 1.0 if (a == 0.0 and b == 0.0) else 0.0

    @classmethod
    def s_region(cls, pred: np.ndarray, gt: np.ndarray) -> float:
        """S_region.m: 4 quadrants about the GT centroid, area-weighted SSIM."""
        h, w = gt.shape
        total = float(gt.sum())
        if total == 0:
            return 0.0
        gtf = gt.astype(np.float64)
        # centroid, MATLAB 1-based
        # MATLAB round() is half-AWAY-from-zero; Python's round() is half-to-EVEN.
        # They disagree on every exact .5 centroid with an even integer part, which
        # shifts the quadrant split by one pixel (audited 2026-09-20: worst case 0.50
        # on 2x2 objects, but 0/180 of this pipeline's silhouette masks ever tie).
        X = _matlab_round(float((np.arange(1, w + 1) * gtf.sum(axis=0)).sum() / total))
        Y = _matlab_round(float((np.arange(1, h + 1) * gtf.sum(axis=1)).sum() / total))
        # DELIBERATE DEVIATION: S_region.m does not clamp, so a centroid on the last
        # row/column gives an empty quadrant, mean2([]) = NaN, and the whole measure
        # returns NaN. We clamp to keep the pipeline finite. Verified never to bind on
        # this corpus (the centroid is interior for every mask used here).
        X = min(max(X, 1), w - 1)
        Y = min(max(Y, 1), h - 1)
        quads = [(slice(0, Y), slice(0, X)), (slice(0, Y), slice(X, w)),
                 (slice(Y, h), slice(0, X)), (slice(Y, h), slice(X, w))]
        area = float(w * h)
        w1 = (X * Y) / area
        w2 = ((w - X) * Y) / area
        w3 = (X * (h - Y)) / area
        w4 = 1.0 - w1 - w2 - w3
        return float(sum(wi * cls._ssim(pred[q].astype(np.float64), gtf[q])
                         for wi, q in zip((w1, w2, w3, w4), quads)))

    @classmethod
    def s_measure(cls, pred: np.ndarray, gt: np.ndarray, alpha: float = 0.5) -> float:
        """StructureMeasure.m"""
        pred = pred.astype(np.float64)
        gtb = gt > 0.5
        y = float(gtb.mean())
        if y == 0.0:
            return float(1.0 - pred.mean())
        if y == 1.0:
            return float(pred.mean())
        q = alpha * cls.s_object(pred, gtb) + (1.0 - alpha) * cls.s_region(pred, gtb)
        return float(max(q, 0.0))

    # -------------------------------------------------------------- E-measure
    @staticmethod
    def _enhanced(binary_pred: np.ndarray, gt: np.ndarray) -> float:
        """Enhancedmeasure.m. Its first line is `FM = logical(FM)`, so the reference
        takes a BINARY prediction. main.m:97-102 binarizes at min(2*mean(cam), 1)
        before calling it. Feeding a continuous sigmoid map here (as the pre-2026-09-20
        code did) is not what the reference computes."""
        pred = binary_pred.astype(np.float64)
        gtf = (gt > 0.5).astype(np.float64)
        h, w = gtf.shape
        s = gtf.sum()
        if s == 0.0:
            enhanced = 1.0 - pred
        elif s == gtf.size:
            enhanced = pred.copy()
        else:
            d_pred = pred - pred.mean()
            d_gt = gtf - gtf.mean()
            align = 2.0 * (d_gt * d_pred) / (d_gt * d_gt + d_pred * d_pred + _EPS)
            enhanced = ((align + 1.0) ** 2) / 4.0
        return float(enhanced.sum() / (w * h - 1 + _EPS))

    @classmethod
    def e_measure(cls, pred: np.ndarray, gt: np.ndarray) -> float:
        """Adaptive E-measure, main.m:97-102: binarize at min(2*mean(pred), 1)."""
        p = pred.astype(np.float64)
        thr = min(2.0 * float(p.mean()), 1.0)
        return cls._enhanced(p > thr, gt)

    # ------------------------------------------------------------- weighted F
    # The distance transform and the importance map depend ONLY on the GT mask, not on
    # the prediction, but the reference recomputes them per call. In this pipeline the
    # same mask is scored 8 times (2 conditions x 2 detectors x 2 normalisations), so a
    # tiny cache keyed on the mask cuts the dominant cost without changing any number.
    _wfb_cache: dict = {}

    @classmethod
    def _wfb_geometry(cls, gtb: np.ndarray):
        key = (gtb.shape, gtb.tobytes())
        hit = cls._wfb_cache.get(key)
        if hit is None:
            dst, idx = _bwdist(~gtb, return_indices=True)
            b = np.where(gtb, 1.0, 2.0 - np.exp(np.log(1 - 0.5) / 5.0 * dst))
            # int32 indices (exact for any real image) halve the entry to ~36 MB,
            # and the bound is 2 because the access pattern is 8 consecutive calls on one
            # mask (2 conditions x 2 detectors x 2 normalisations) before the mask changes.
            hit = (idx.astype(np.int32), b)   # int32 indices are exact; b stays float64
            if len(cls._wfb_cache) >= 2:
                cls._wfb_cache.clear()
            cls._wfb_cache[key] = hit
        return hit

    @classmethod
    def weighted_f_measure(cls, pred: np.ndarray, gt: np.ndarray, beta: float = 1.0) -> float:
        """original_WFb.m (Margolin et al. 2014)."""
        pred = pred.astype(np.float64)
        gtb = gt > 0.5
        if not gtb.any():
            return 0.0
        e = np.abs(pred - gtb.astype(np.float64))
        idx, b = cls._wfb_geometry(gtb)
        # Et: pixel dependency - background error inherits its nearest GT pixel's error
        et = e.copy()
        et[~gtb] = e[idx[0][~gtb], idx[1][~gtb]]
        k = cv2.getGaussianKernel(7, 5.0)
        ea = cv2.filter2D(et, -1, k @ k.T, borderType=cv2.BORDER_CONSTANT)
        min_e_ea = e.copy()
        sel = gtb & (ea < e)
        min_e_ea[sel] = ea[sel]
        ew = min_e_ea * b          # b: pixel importance, from the cached geometry
        tpw = float(gtb.sum()) - float(ew[gtb].sum())
        fpw = float(ew[~gtb].sum())
        r = 1.0 - float(ew[gtb].mean())
        p = tpw / (tpw + fpw + _EPS)
        b2 = beta ** 2
        return float((1.0 + b2) * r * p / (b2 * p + r + _EPS))

    # ------------------------------------------------------------------ all
    @classmethod
    def evaluate_all(cls, pred: np.ndarray, gt: np.ndarray) -> dict:
        p = pred.astype(np.float64)
        g = gt.astype(np.float64)
        if p.max() > 1.01:
            p /= 255.0
        if g.max() > 1.01:
            g /= 255.0
        # main.m:89 min-max stretches every prediction to [0,1] before scoring.
        # That convention makes a uniformly-near-zero map (a total miss) look like a
        # confident detection, which is exactly the wrong behaviour for camouflage
        # evaluation, so the UNNORMALISED values are primary here and the benchmark
        # convention is reported alongside for comparability.
        rng = float(p.max() - p.min())
        pn = (p - p.min()) / rng if rng > 1e-12 else np.zeros_like(p)
        return {
            'MAE':          cls.mae(p, g),
            'S_alpha':      cls.s_measure(p, g),
            'E_phi':        cls.e_measure(p, g),
            'Fw_beta':      cls.weighted_f_measure(p, g),
            'S_alpha_norm': cls.s_measure(pn, g),
            'E_phi_norm':   cls.e_measure(pn, g),
            'Fw_beta_norm': cls.weighted_f_measure(pn, g),
        }

### The two detectors — Evaluation, "Digital Evaluation"

SINet-V2 and DGNet act as fixed judges. Both are used exactly as published, with no retraining; their code and weights are in `models/`.

In [ ]:
%%module detectors
"""The two camouflaged object detectors, used as published with no retraining.

    SINet-V2  Fan et al. 2022, IEEE TPAMI   https://github.com/GewelsJI/SINet-V2
    DGNet     Ji et al. 2023, MIR           https://github.com/GewelsJI/DGNet (EfficientNet-B4)

Each detector takes an RGB uint8 image and returns a float32 map in [0, 1] at the image
size: the sigmoid of its output after a resize of the input to 352 x 352.

torch loads only when build_detectors() runs. When torch and torchvision load before the
k-means palette step in the same process, the palette changes (3 pixels of the M81
pattern in a test on 2026-09-28), so the import must stay inside the function.
"""
import sys, warnings
from pathlib import Path

import numpy as np

ROOT = Path.cwd()          # project root (the notebook starts here)
SINET_WEIGHTS = ROOT / 'models' / 'SINet-V2' / 'snapshot' / 'SINet_V2' / 'Net_epoch_best.pth'
DGNET_WEIGHTS = ROOT / 'models' / 'DGNet' / 'lib_pytorch' / 'snapshot' / 'DGNet.pth'


def _import_repo(repo_dir):
    """Both repositories have a top-level package `lib`. Remove the one that is loaded and
    put this repository first on the path."""
    for mod in [m for m in list(sys.modules) if m == 'lib' or m.startswith('lib.')]:
        del sys.modules[mod]
    sys.path.insert(0, str(repo_dir))


class Detector:
    def __init__(self, model, device, pick):
        import torch
        import torchvision.transforms as T
        self.torch, self.model, self.device, self.pick = torch, model.to(device).eval(), device, pick
        self.transform = T.Compose([T.ToPILImage(), T.Resize((352, 352)), T.ToTensor(),
                                    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    def predict(self, image):
        torch = self.torch
        h, w = image.shape[:2]
        x = self.transform(image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            pred = torch.sigmoid(self.pick(self.model(x)))
            pred = torch.nn.functional.interpolate(pred, size=(h, w), mode='bilinear', align_corners=True)
        return pred.squeeze().cpu().numpy().astype(np.float32)


def build_detectors(device):
    """Returns (SINet-V2, DGNet) on `device` ('mps' or 'cpu')."""
    import torch
    device = torch.device(device)

    _import_repo(SINET_WEIGHTS.resolve().parents[2])
    from lib.Network_Res2Net_GRA_NCD import Network
    sinet = Network(channel=32, imagenet_pretrained=False)
    sinet.load_state_dict(torch.load(str(SINET_WEIGHTS), map_location=device, weights_only=False))
    print(f'[SINetV2] Official model loaded — device={device}')

    _import_repo(DGNET_WEIGHTS.resolve().parents[1])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        from lib.DGNet import DGNet
    dgnet = DGNet(channel=64, arc='EfficientNet-B4', M=[8, 8, 8], N=[4, 8, 16])
    # strict=False: the two extra keys are the classification head of the backbone, which DGNet does not use
    missing, _ = dgnet.load_state_dict(torch.load(str(DGNET_WEIGHTS), map_location=device, weights_only=False), strict=False)
    assert not missing, f'DGNet: {len(missing)} missing keys'
    print(f'[DGNet] Model loaded (EfficientNet-B4, channel=64) — device={device}')

    # SINet-V2 returns several maps, the last is the prediction; DGNet returns (prediction, gradient map)
    return (Detector(sinet, device, lambda out: out[-1] if isinstance(out, (list, tuple)) else out),
            Detector(dgnet, device, lambda out: out[0] if isinstance(out, (tuple, list)) else out))

### Compositing and the detector run — Evaluation, "Compositing Protocol"

Each test pastes one silhouette into a test scene, fills it with a pattern, and asks both detectors to find it. The pattern is tiled so that one repeat is one third of the silhouette height. Each silhouette is moved by up to 10 % of the frame in 20 positions, and both conditions use the same positions. The base pattern is first cropped to the shape of the site-specific pattern and reduced to the same 400×300 size, so that the two differ in design and not in image size.

In [ ]:
%%module adversarial_eval
"""Digital evaluation: paste a silhouette into a test scene, fill it with a pattern, and ask
both detectors to find it.

Compositing protocol: the pattern is tiled so that one repeat is one third of the
silhouette height, and the silhouette edge is feathered (Gaussian, sigma 3 px). Each
silhouette is moved by up to 10 % of the frame in 20 positions. The positions come from a
seed of (base, scene, shape), so the site-specific pattern and its base pattern, and every
arm of the color-limit test, use the same 20 positions. The base pattern is first cropped to
the shape of the site-specific pattern and reduced to its 400 x 300 size
(equal_resolution_base), so the two differ in design and not in image size.

run_eval() scores the base pattern ('generic') and the site-specific pattern ('site') on
every test scene and writes one row per detector run to results.csv. An interrupted run
continues where it stopped.
"""
import csv
import hashlib
import time
from pathlib import Path

import cv2
import numpy as np

from silhouettes import SilhouetteLibrary
from cod_metrics import CODMetrics
from detectors import build_detectors

ROOT = Path.cwd()          # project root (the notebook starts here)
CAMO_ROOT = ROOT / 'results' / '01_generation' / 'patterns'
EVAL_ROOT = ROOT / 'results' / '02_adversarial_test'
BASE_DIR  = ROOT / 'images' / 'base_patterns'

PLACEMENT_FRAC = 0.10     # largest move of the silhouette, as a fraction of the frame
REPEAT_FRAC    = 1/3      # height of one pattern repeat, as a fraction of the silhouette height
METRICS        = ['S_alpha', 'E_phi', 'Fw_beta', 'MAE', 'S_alpha_norm', 'E_phi_norm', 'Fw_beta_norm']
# detector output inside and outside the target, and its highest value
EXTRA_COLS     = ['pred_in_gt', 'pred_out_gt', 'pred_max']
# ColorChecker photographs: calibration inputs, never test scenes
CHART_FRAMES = {'IMG_4174', 'IMG_4193', 'IMG_4194', 'IMG_4198', 'IMG_4204', 'IMG_4209'}


def seed_of(*parts):
    return int(hashlib.md5('|'.join(parts).encode()).hexdigest(), 16) % (2 ** 32)


def load_rgb(p):
    """Image file -> RGB uint8 (16-bit, gray and 4-channel files are converted)."""
    bgr = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if bgr is None:
        raise FileNotFoundError(p)
    if bgr.dtype == np.uint16:
        bgr = (bgr / 257.0).round().astype(np.uint8)
    if bgr.ndim == 2:
        bgr = cv2.cvtColor(bgr, cv2.COLOR_GRAY2BGR)
    if bgr.shape[2] == 4:
        bgr = bgr[..., :3]
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def composite(camo, bg, mask):
    """Tile `camo` into the silhouette `mask` on the scene `bg`. Returns the image and the
    target mask (float, 1 = target)."""
    h, w = bg.shape[:2]
    sil_h = max(int((mask > 127).any(axis=1).sum()), 1)   # rows that hold the silhouette
    th = max(int(sil_h * REPEAT_FRAC), 8)
    tw = max(int(th * camo.shape[1] / camo.shape[0]), 8)  # keep the aspect ratio of the pattern
    camo = np.tile(cv2.resize(camo, (tw, th)), (h // th + 1, w // tw + 1, 1))[:h, :w]
    a = cv2.GaussianBlur(mask.astype(np.float32), (0, 0), 3.0)
    a = a / max(a.max(), 1e-6)
    out = (a[..., None] * camo + (1 - a[..., None]) * bg).astype(np.uint8)
    return out, (mask > 127).astype(np.float32)


def equal_resolution_base(style, frame='IMG_4171', tag='_cal'):
    """The base pattern cropped to the shape of the style-transfer output and reduced to its
    size (400 x 300, cv2.INTER_AREA), so the two patterns differ in design and not in image size."""
    h, w = cv2.imread(str(CAMO_ROOT / style / (frame + tag) / 'final_camo.png')).shape[:2]
    base = cv2.cvtColor(cv2.imread(str(BASE_DIR / f'{style}.jpg')), cv2.COLOR_BGR2RGB)
    bh, bw = base.shape[:2]
    cw = min(bw, round(bh * w / h)); ch = min(bh, round(cw * h / w))
    y0, x0 = (bh - ch) // 2, (bw - cw) // 2
    return cv2.resize(base[y0:y0 + ch, x0:x0 + cw], (w, h), interpolation=cv2.INTER_AREA)


def placements(style, stem, shape, n_trials, h, w):
    """The n_trials (dx, dy) moves of one silhouette in one scene."""
    rng = np.random.RandomState(seed_of(style, stem, shape))
    max_dx, max_dy = int(w * PLACEMENT_FRAC), int(h * PLACEMENT_FRAC)
    return [(int(rng.randint(-max_dx, max_dx + 1)), int(rng.randint(-max_dy, max_dy + 1))) for _ in range(n_trials)]


def test_scenes(png_dir, frame):
    """All calibrated PNGs except the chart photographs and the generation frame."""
    stems = sorted(p.stem for p in Path(png_dir).glob('*.png'))
    return [s for s in stems if s not in CHART_FRAMES and s != frame]


def run_eval(frame, styles, n_trials, device, png_dir, base_path, tag='_cal', bg_limit=None):
    """Score the base pattern (file base_path(style)) and the site-specific pattern of each
    style on every test scene. Writes EVAL_ROOT/<frame><tag>/results.csv."""
    out = EVAL_ROOT / (frame + tag)
    out.mkdir(parents=True, exist_ok=True)
    csvp, prog = out / 'results.csv', out / 'progress.txt'

    def log(m):
        print(m, flush=True)
        with open(prog, 'a') as f:
            f.write(m + '\n')

    t0 = time.time()
    eval_stems = test_scenes(png_dir, frame)[:bg_limit]
    camos = {s: {'site': load_rgb(CAMO_ROOT / s / (frame + tag) / 'final_camo.png'),
                 'generic': load_rgb(base_path(s))} for s in styles}
    sinet, dgnet = build_detectors(device)
    detectors = [('SINet-V2', sinet), ('DGNet', dgnet)]
    shapes = SilhouetteLibrary.CLASSES
    log(f'[start] {len(styles)} bases x {len(eval_stems)} scenes x {len(shapes)} shapes x {n_trials} positions '
        f'x 2 conditions x 2 detectors, device={device}')

    fields = ['style', 'bg', 'shape', 'trial', 'dx', 'dy', 'condition', 'detector'] + METRICS + EXTRA_COLS
    done = {(r['style'], r['bg'], r['shape']) for r in csv.DictReader(open(csvp))} if csvp.exists() else set()
    if done:
        log(f'[resume] {len(done)} cells already complete')
    else:
        with open(csvp, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=fields).writeheader()

    cell, n_cells = 0, len(styles) * len(eval_stems) * len(shapes)
    for style in styles:
        for stem in eval_stems:
            bg = load_rgb(Path(png_dir) / f'{stem}.png')
            h, w = bg.shape[:2]
            for shape in shapes:
                cell += 1
                if (style, stem, shape) in done:
                    continue
                base_mask = SilhouetteLibrary.get(shape, h, w)
                buf = []
                for t, (dx, dy) in enumerate(placements(style, stem, shape, n_trials, h, w)):
                    mask = np.roll(np.roll(base_mask, dy, 0), dx, 1)   # the same position for both conditions
                    for cond in ('generic', 'site'):
                        comp, gt = composite(camos[style][cond], bg, mask)
                        gtb = gt > 0.5
                        for dn, det in detectors:
                            pred = det.predict(comp)
                            m = CODMetrics.evaluate_all(pred, gt)
                            row = {'style': style, 'bg': stem, 'shape': shape, 'trial': t,
                                   'dx': dx, 'dy': dy, 'condition': cond, 'detector': dn}
                            row.update({k: float(m[k]) for k in METRICS})
                            row['pred_in_gt'] = float(pred[gtb].mean())
                            row['pred_out_gt'] = float(pred[~gtb].mean())
                            row['pred_max'] = float(pred.max())
                            buf.append(row)
                with open(csvp, 'a', newline='') as f:
                    csv.DictWriter(f, fieldnames=fields).writerows(buf)
                if cell % 6 == 0 or cell == n_cells:
                    log(f'[eval] cell {cell}/{n_cells} ({style}) +{(time.time()-t0)/60:.1f} min')
    log(f'[eval-done] +{(time.time()-t0)/60:.1f} min -> {csvp}')

### Tables for the paper

Each function reads a result file and returns one table of the paper in LaTeX. Nothing in a table is typed by hand.

In [ ]:
%%module paper_tables
"""LaTeX tables for the paper, in the format of the manuscript, made from the saved result files.

Each function reads a CSV that a notebook wrote, returns the LaTeX of one table, and
writes it to `out` when `out` is given. Nothing in a table is typed by hand.
"""
import csv
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy import stats

STYLES = ['multicam', 'woodland', 'realtree']
NAME = {'multicam': 'MultiCam', 'woodland': 'M81', 'realtree': 'Realtree'}   # 'woodland' is the M81 base
DETECTORS = ('SINet-V2', 'DGNet')


def _holm(ps):
    order = sorted(range(len(ps)), key=lambda i: ps[i])
    adj, prev = [0.0] * len(ps), 0.0
    for rank, i in enumerate(order):
        prev = max(prev, min(1.0, (len(ps) - rank) * ps[i]))
        adj[i] = prev
    return adj


def _write(tex, out):
    if out:
        out = Path(out)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(tex + '\n')
    return tex


def quant_table(table_csv, out=None):
    """Table tab:quant from colour_limit/table_quant_ablation.csv (notebook 1)."""
    rows = [r for r in csv.DictReader(open(table_csv)) if r['metric'] == 'S_alpha']
    get = lambda st, det, c: next(r for r in rows if r['style'] == st and r['detector'] == det and r['contrast'] == c)
    cells = [(st, det) for st in STYLES for det in DETECTORS]
    hp = dict(zip(cells, _holm([float(get(st, det, 'dq_nst-dq_nonst')['p_two_sided']) for st, det in cells])))
    hw = dict(zip(cells, _holm([float(get(st, det, 'k5-kfull')['p_two_sided']) for st, det in cells])))
    ho = dict(zip(cells, _holm([float(get(st, det, 'nonst_k5-nonst_kfull')['p_two_sided']) for st, det in cells])))
    star = lambda v, p: f'${v:+.3f}^{{*}}$' if p < 0.05 else f'${v:+.3f}$'
    body = []
    for st in STYLES:
        for det in DETECTORS:
            w, wo, d = (get(st, det, c) for c in ('k5-kfull', 'nonst_k5-nonst_kfull', 'dq_nst-dq_nonst'))
            dv = float(d['delta'])
            # bold = reduction significant after Holm; * = dq significant after Holm (per column)
            cell = rf'$\mathbf{{{dv:+.3f}}}$' if dv < 0 and hp[(st, det)] < 0.05 else f'${dv:+.3f}$'
            body.append(f"{NAME[st]:8} & {det:8} & {star(float(w['delta']), hw[(st, det)])} & "
                        f"{star(float(wo['delta']), ho[(st, det)])} & {cell} \\\\")
    return _write(r'''\begin{table}[t]
\centering
\caption{The cost of the five-color limit. Each row is named after the base pattern; the values are about the site-specific pattern made from it. $\Delta_q$ is the $S_\alpha$ score
with five colors minus the score with no color limit; a positive value means
five colors make the pattern easier to find. Each value is the mean of the 33
scene-level differences. ``$\Delta_q$ with'' uses the
style-transfer pattern; ``$\Delta_q$ without'' repeats the test on the base pattern
alone, at the same image size. The difference is with minus without; a negative
value means style transfer reduces the cost. Two-sided Wilcoxon tests over 33
scenes, Holm correction over the six cells of each column: an asterisk marks a
significant $\Delta_q$, bold a significant reduction.}
\label{tab:quant}
\small
\setlength{\tabcolsep}{4pt}
\begin{tabular}{llrrr}
\hline
Base & Detector & $\Delta_q$ with & $\Delta_q$ without & difference \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)


def base_table(results_csv, out=None):
    """Table tab:base from 02_adversarial_test/IMG_4171_cal/results.csv (notebook 2)."""
    agg = defaultdict(list)
    for r in csv.DictReader(open(results_csv)):
        agg[(r['bg'], r['style'], r['detector'], r['shape'], r['condition'])].append(float(r['S_alpha']))
    bgs = sorted({k[0] for k in agg}); shapes = sorted({k[3] for k in agg})
    cells = [(st, det) for st in STYLES for det in DETECTORS]
    D = {(st, det): np.array([np.mean([np.mean(agg[(b, st, det, sh, 'site')]) - np.mean(agg[(b, st, det, sh, 'generic')])
                                       for sh in shapes]) for b in bgs]) for st, det in cells}
    H = dict(zip(cells, _holm([stats.wilcoxon(D[c]).pvalue for c in cells])))
    holm_note = ("``better'' and ``worse'' are significant after Holm correction over the six tests; "
                 "``no difference'' is not")
    body = []
    for st, det in cells:
        d = D[(st, det)]
        res = ('no difference' if H[(st, det)] >= 0.05 else
               'better' if d.mean() < 0 else r'\textbf{worse}')
        hp_ = H[(st, det)]
        ptxt = '$<0.001$' if hp_ < 0.001 else f'${hp_:.2f}$'
        body.append(f"{NAME[st]:8} & {det:8} & ${d.mean():+.3f}$ & {int((d < 0).sum())}/{len(d)} & {ptxt} & {res} \\\\")
    return _write(r'''\begin{table}[t]
\centering
\caption{Each site-specific pattern against the base pattern it was made from.
Each row is named after the base pattern. $\Delta S_\alpha$ is the mean of the 33 scene-level differences, site-specific
minus base, so a negative value means the site-specific pattern is harder to find. The base is downsampled to the size of the
site-specific pattern. ``Wins'' counts how many of the 33
scenes favor it. Tests are two-sided Wilcoxon signed-rank; ''' + holm_note + r'''.}
\label{tab:base}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{llrrrl}
\hline
Base & Detector & $\Delta S_\alpha$ & Wins & Holm $p$ & Result \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)


FIELD_LABEL = {'print_v1': r'\texttt{print\_v1} (pipeline)', 'print_v2': r'\texttt{print\_v2}',
               'multicam_ocp': 'MultiCam', 'woodland_m81': 'M81, view 1', 'woodland_m81_b': 'M81, view 2',
               'hunting_realtree': 'Realtree'}
FIELD_ORDER = ['print_v1', 'print_v2', 'multicam_ocp', 'woodland_m81', 'woodland_m81_b', 'hunting_realtree']


def location_table(results_csv, site, out=None):
    """Table tab:locA (site1) or tab:locB (site2) from 03_field_test/two_sites/results.csv (notebook 3)."""
    rows = [r for r in csv.DictReader(open(results_csv)) if r['site'] == site]
    fmt = lambda v: '$<0.01$' if v < 0.01 else f'${v:.2f}$'
    body = []
    for label in [l for l in FIELD_ORDER if any(r['label'] == l for r in rows)]:
        vals = []
        for kind in ('full', 'best'):
            for det in DETECTORS:
                r = [float(x['pred_in_gt']) for x in rows if x['label'] == label and x['detector'] == det
                     and (kind == 'best' or x['condition'] == 'full')]
                vals.append(fmt(max(r)))
        name = FIELD_LABEL[label] if site == 'site2' else FIELD_LABEL[label].replace(', view 1', '')
        body.append(f'{name} & ' + ' & '.join(vals) + r' \\')
    n = 'AB'[int(site[-1]) - 1]
    if site == 'site1':
        cap = (r'Location~A. Mean detector score inside the traced outline; lower means harder to find. '
               r"``Max'' is the highest detector score over the full frame and crops of $1.5\times$, $2.5\times$ and "
               r'$4\times$ the item size. \texttt{print\_v2} is not pipeline output.')
    else:
        cap = (r'Location~B, same measures as Table~\ref{tab:locA}. The two M81 rows are two photographs '
               r'of one unmoved drape, taken 17\,s apart.')
    return _write(r'''\begin{table}[t]
\centering
\caption{''' + cap + r'''}
\label{tab:loc''' + n + r'''}
\footnotesize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lrrrr}
\hline
 & \multicolumn{2}{c}{Full frame} & \multicolumn{2}{c}{Max} \\
Item & SINet-V2 & DGNet & SINet-V2 & DGNet \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)


def recolor_table(site_csv, ablation_csv, out=None):
    """Table tab:recolor: how much color alone does (notebook 2, uses the notebook 1 ablation rows).

    recolored = base pattern recolored with the five site colors, no style transfer (nonst_k5);
    all arms share scenes, shapes and positions, so each column is a paired difference."""
    agg = defaultdict(list)
    for f in (site_csv, ablation_csv):
        for r in csv.DictReader(open(f)):
            if r['condition'] in ('site', 'generic', 'nonst_k5', 'kfull'):
                agg[(r['bg'], r['style'], r['detector'], r['shape'], r['condition'])].append(float(r['S_alpha']))
    bgs = sorted({k[0] for k in agg}); shapes = sorted({k[3] for k in agg})
    vec = lambda st, det, c: np.array([np.mean([np.mean(agg[(b, st, det, sh, c)]) for sh in shapes]) for b in bgs])
    cells = [(st, det) for st in STYLES for det in DETECTORS]
    cols = [('nonst_k5', 'generic'), ('site', 'nonst_k5'), ('kfull', 'generic')]
    D = {(c, k): vec(*c, k[0]) - vec(*c, k[1]) for c in cells for k in cols}
    H = {k: dict(zip(cells, _holm([stats.wilcoxon(D[(c, k)]).pvalue for c in cells]))) for k in cols}
    body = []
    for c in cells:
        vals = [f'${D[(c, k)].mean():+.3f}^{{*}}$' if H[k][c] < 0.05 else f'${D[(c, k)].mean():+.3f}$' for k in cols]
        body.append(f"{NAME[c[0]]:8} & {c[1]:8} & " + ' & '.join(vals) + r' \\')
    return _write(r'''\begin{table}[t]
\centering
\caption{What color alone does. Each row is named after the base pattern. Each column is a paired $\Delta S_\alpha$ on the same
33 scenes and positions, given as the mean of the 33 scene-level differences; a negative value means the first pattern is harder to
find. ``Recolored'' is the base pattern recolored with the five site colors, with
no style transfer; ``site'' is the site-specific pattern; ``no limit'' is the
style-transfer pattern colored with no color limit. An asterisk marks a
difference that is significant after Holm correction over the six cells of its
column (two-sided Wilcoxon).}
\label{tab:recolor}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{llrrr}
\hline
 & & recolored & site & no limit \\
Base & Detector & $-$ base & $-$ recolored & $-$ base \\
\hline
''' + '\n'.join(body) + r'''
\hline
\end{tabular}
\end{table}''', out)

## Detector run

The base condition uses the base pattern at the size of the site-specific pattern (`equal_resolution_base`). `run_eval` appends to `results.csv` as it goes, so with `FRESH_RUN = False` an interrupted run continues. A continued run must use the same patterns: the cell records a hash of each pattern file and stops if the patterns changed since the rows on disk were written.

In [ ]:
import adversarial_eval as rv
EVAL.mkdir(parents=True, exist_ok=True)
stamp = EVAL/'pattern_hashes.json'
now = {s: hashlib.sha256((PATTERNS/s/(FRAME+TAG)/'final_camo.png').read_bytes()).hexdigest() for s in STYLES}
if stamp.exists() and json.load(open(stamp)) != now:
    raise RuntimeError('the patterns changed since results.csv was started; set FRESH_RUN = True and start again')
json.dump(now, open(stamp, 'w'), indent=1)

EQ_BASES = OUT/'equal_resolution_bases'; EQ_BASES.mkdir(parents=True, exist_ok=True)
for s in STYLES:
    cv2.imwrite(str(EQ_BASES/f'{s}.png'), cv2.cvtColor(rv.equal_resolution_base(s), cv2.COLOR_RGB2BGR))
rv.run_eval(FRAME, STYLES, N_TRIALS, DEVICE, PNG_DIR, base_path=lambda s: EQ_BASES/f'{s}.png', tag=TAG, bg_limit=BG_LIMIT)

## Table `tab:base` — each site-specific pattern against its base pattern

Shapes and positions are averaged within each scene, so the paired units are the 33 scenes. $\Delta S_\alpha$ = site-specific − base. "Wins" = scenes on which the site-specific pattern has the lower $S_\alpha$. Two-sided Wilcoxon signed-rank test; Holm correction over the six tests.

In [ ]:
rows = list(csv.DictReader(open(EVAL/'results.csv')))
bgs = sorted({r['bg'] for r in rows}); shapes = sorted({r['shape'] for r in rows})
print(f'{len(rows)} detector rows, {len(bgs)} backgrounds, shapes {shapes}\n')

def cell_means(metric):
    agg = defaultdict(list)
    for r in rows:
        agg[(r['bg'], r['style'], r['detector'], r['shape'], r['condition'])].append(float(r[metric]))
    return {k: float(np.mean(v)) for k, v in agg.items()}
SA = cell_means('S_alpha')

def per_bg(style, det, shape=None):
    """per-scene S_alpha of the site-specific pattern minus its base pattern (mean over the shapes, or one shape)"""
    shs = [shape] if shape else shapes
    return np.array([np.mean([SA[(bg, style, det, sh, 'site')] - SA[(bg, style, det, sh, 'generic')] for sh in shs]) for bg in bgs])

def holm(ps):
    order = sorted(range(len(ps)), key=lambda i: ps[i]); adj = [0.0]*len(ps); prev = 0.0
    for rank, i in enumerate(order):
        prev = max(prev, min(1.0, (len(ps) - rank) * ps[i])); adj[i] = prev
    return adj

cells = [(st, det) for st in STYLES for det in DETECTORS]
D = {c: per_bg(*c) for c in cells}
P = {c: stats.wilcoxon(D[c]).pvalue for c in cells}
H = dict(zip(cells, holm([P[c] for c in cells])))
print(f"{'base':9}{'detector':10}{'dS_alpha':>9}{'wins':>8}{'p':>10}{'Holm':>10}  result")
for c in cells:
    d = D[c]
    print(f"{NAME[c[0]]:9}{c[1]:10}{d.mean():+9.3f}{int((d < 0).sum()):>5}/{len(d)}{P[c]:10.1e}{H[c]:10.1e}  "
          f"{'better' if d.mean() < 0 else 'WORSE'}{'' if H[c] < 0.05 else ' (not significant after Holm)'}"
          f"{'  |d| > eps' if abs(d.mean()) > EPS else ''}")

### Split by target shape, and by location

The paper reports that four of the eighteen shape cells go the wrong way, and that splitting the scenes by location keeps the sign of every cell.

In [ ]:
print('dS_alpha by shape (a positive value means the site-specific pattern is easier to find):')
print(f"{'base':9}{'detector':10}" + ''.join(f'{sh:>11}' for sh in shapes))
rev = 0
for c in cells:
    vals = [per_bg(*c, shape=sh).mean() for sh in shapes]
    rev += sum(v > 0 for v in vals)
    print(f"{NAME[c[0]]:9}{c[1]:10}" + ''.join(f"{v:+10.3f}{'*' if v > 0 else ' '}" for v in vals))
print(f'{rev} of {len(cells)*len(shapes)} shape cells reverse (marked *)\n')

loc_a = [i for i, bg in enumerate(bgs) if bg in LOCATION_A]
loc_b = [i for i, bg in enumerate(bgs) if bg not in LOCATION_A]
print(f'dS_alpha by location: A = generation location ({len(loc_a)} backgrounds), B = 11.3 km away ({len(loc_b)} backgrounds)')
same = 0
for c in cells:
    a, b = D[c][loc_a].mean(), D[c][loc_b].mean()
    same += np.sign(a) == np.sign(D[c].mean()) == np.sign(b)
    print(f"  {NAME[c[0]]:9}{c[1]:10} A {a:+.3f}   B {b:+.3f}   all {D[c].mean():+.3f}")
print(f'sign of the pooled result kept at both locations in {same} of {len(cells)} cells')

### The same test with the base pattern at its original size

The paper states that with the base pattern at its original 1600×1600 size, five cells favored the site-specific pattern. That run (2026-09-24, commit `c609b81`) is kept in `results/native_resolution_2026-09-24/`.

In [ ]:
agg_n = defaultdict(list)
for r in csv.DictReader(open(ROOT/'results'/'native_resolution_2026-09-24'/'adversarial_results.csv')):
    agg_n[(r['bg'], r['style'], r['detector'], r['shape'], r['condition'])].append(float(r['S_alpha']))
Dn = {c: np.array([np.mean([np.mean(agg_n[(b, c[0], c[1], sh, 'site')]) - np.mean(agg_n[(b, c[0], c[1], sh, 'generic')])
                            for sh in shapes]) for b in bgs]) for c in cells}
print(f"{'base':9}{'detector':10}{'dS_alpha: equal':>17}{'wins':>7}{'native':>9}{'wins':>7}")
for c in cells:
    e, n = D[c], Dn[c]
    print(f"{NAME[c[0]]:9}{c[1]:10}{e.mean():+17.3f}{int((e < 0).sum()):>4}/33{n.mean():+9.3f}{int((n < 0).sum()):>4}/33"
          f"{'   sign changed' if e.mean() * n.mean() < 0 else ''}")

## Figure `fig_composites` for the paper

One test scene (`IMG_4172`) with the human silhouette filled with each base pattern (top) and the site-specific pattern made from it (bottom), cropped around the silhouette. The pattern is tiled as in the test, and the base is used at the size of the site-specific pattern.

In [ ]:
from silhouettes import SilhouetteLibrary
SCENE = 'IMG_4172'
EQ_BASES = OUT/'equal_resolution_bases'
bg = rv.load_rgb(PNG_DIR/f'{SCENE}.png'); h, w = bg.shape[:2]
mask = SilhouetteLibrary.get('human', h, w)
ys, xs = np.where(mask > 127)
cy, cx = (ys.min() + ys.max()) // 2, (xs.min() + xs.max()) // 2
half = int((ys.max() - ys.min()) * 0.75)                       # crop: silhouette plus ground around it
y0, y1 = max(0, cy - half), min(h, cy + half); x0, x1 = max(0, cx - int(half * 0.75)), min(w, cx + int(half * 0.75))
fig, ax = plt.subplots(2, 3, figsize=(7.2, 6.4))
for j, st in enumerate(('multicam', 'woodland', 'realtree')):
    for i, (label, p) in enumerate([('base', EQ_BASES/f'{st}.png'), ('site-specific', PATTERNS/st/(FRAME+TAG)/'final_camo.png')]):
        comp, _ = rv.composite(rv.load_rgb(p), bg, mask)
        ax[i, j].imshow(comp[y0:y1, x0:x1]); ax[i, j].axis('off')
        ax[i, j].set_title(f'{NAME[st]} {label}', fontsize=9)
plt.tight_layout(); plt.savefig(FIG/'fig_composites.png', dpi=200, bbox_inches='tight'); plt.show()
print('->', FIG/'fig_composites.png')

## Tables `tab:base` and `tab:recolor` for the paper

The cell writes the LaTeX of the tables to `results/02_adversarial_test/tables/`. `tab:recolor` ("What color alone does") also uses the color-limit results of notebook 1.

In [ ]:
import paper_tables as pt
TAB = OUT/'tables'
print(pt.base_table(EVAL/'results.csv', out=TAB/'tab_base.tex'))
print(pt.recolor_table(EVAL/'results.csv', ROOT/'results'/'01_generation'/'colour_limit'/'results.csv', out=TAB/'tab_recolor.tex'))

## Answer — hypothesis half (b)

This cell applies the rule from the top of the notebook to the table `tab:base`.

In [ ]:
ALPHA = 0.05
print('HYPOTHESIS HALF (b) - advantage over the base pattern each design was made from')
better = [c for c in cells if D[c].mean() < 0 and H[c] < ALPHA]
worse  = [c for c in cells if D[c].mean() > 0 and H[c] < ALPHA]
for c in cells:
    d = D[c].mean()
    v = 'supported' if c in better else ('significant in the WRONG direction' if c in worse else 'no significant difference')
    print(f"  {NAME[c[0]]:9}{c[1]:10} dS_alpha {d:+.3f}  wins {int((D[c] < 0).sum())}/{len(D[c])}  Holm p {H[c]:.1e}  "
          f"{'|d| > eps' if abs(d) > EPS else '|d| <= eps'}  -> {v}")
print(f"ANSWER: half (b) is supported in {len(better)} of {len(cells)} cells"
      + (f"; {len(worse)} cell(s) go the wrong way ({', '.join(NAME[c[0]] + '/' + c[1] for c in worse)})" if worse else '') + '.')